In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [9]:
for root, dirs, files in os.walk("../data/test"):
    for f in files:
        print(f"{root}/{f}")

../data/test\galaxy/100.jpg
../data/test\galaxy/1000.jpg
../data/test\galaxy/1001.jpg
../data/test\galaxy/1002.jpg
../data/test\galaxy/1003.jpg
../data/test\galaxy/1004.jpg
../data/test\galaxy/1005.jpg
../data/test\galaxy/1006.jpg
../data/test\galaxy/1007.jpg
../data/test\galaxy/1008.jpg
../data/test\galaxy/1009.jpg
../data/test\galaxy/101.jpg
../data/test\galaxy/1010.jpg
../data/test\galaxy/1011.jpg
../data/test\galaxy/1012.jpg
../data/test\galaxy/1013.jpg
../data/test\galaxy/1014.jpg
../data/test\galaxy/1015.jpg
../data/test\galaxy/1016.jpg
../data/test\galaxy/1017.jpg
../data/test\galaxy/1018.jpg
../data/test\galaxy/1019.jpg
../data/test\galaxy/102.jpg
../data/test\galaxy/1020.jpg
../data/test\galaxy/1021.jpg
../data/test\galaxy/1022.jpg
../data/test\galaxy/1023.jpg
../data/test\galaxy/1025.jpg
../data/test\galaxy/1026.jpg
../data/test\galaxy/1027.jpg
../data/test\galaxy/1028.jpg
../data/test\galaxy/1029.jpg
../data/test\galaxy/103.jpg
../data/test\galaxy/1030.jpg
../data/test\galax

In [8]:
TRAIN_DIR = "../data/train"
TEST_DIR = "../data/test"
RESULT_DIR = "../results"
os.makedirs(RESULT_DIR, exist_ok=True)
 
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

In [3]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)
 
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 92259 files belonging to 5 classes.
Found 2686 files belonging to 5 classes.


In [12]:
class_names = train_ds.class_names
num_classes = len(class_names)
 
def flatten_dataset(dataset):
    X = []
    y = []
    for batch, labels in dataset:
        batch_np = batch.numpy() / 255.0
        X.append(batch_np.reshape(batch_np.shape[0], -1))
        y.append(labels.numpy())
    return np.vstack(X), np.hstack(y)
 
X_train, y_train = flatten_dataset(train_ds)
X_test, y_test = flatten_dataset(test_ds)

MemoryError: Unable to allocate 16.9 GiB for an array with shape (92259, 49152) and data type float32

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
 
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)
 
learning_rates = [0.001, 0.0005]
batch_sizes = [32, 64]

In [ ]:
learning_rates = [0.001, 0.0005]
batch_sizes = [32, 64]
 
best_acc = 0
best_model = None
best_params = None
 
for lr in learning_rates:
    for bs in batch_sizes:
 
        model = Sequential([
            Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation='relu'),
            BatchNormalization(),
            Dropout(0.3),
            Dense(128, activation='relu'),
            BatchNormalization(),
            Dense(num_classes, activation='softmax')
        ])
 
        model.compile(optimizer=Adam(learning_rate=lr),
                      loss="categorical_crossentropy",
                      metrics=["accuracy"])
 
        es = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
 
        model.fit(
            X_train, y_train_cat,
            epochs=20,
            batch_size=bs,
            validation_split=0.2,
            callbacks=[es],
            verbose=0
        )
 
        y_pred = model.predict(X_test)
        y_pred_labels = np.argmax(y_pred, axis=1)
        acc = accuracy_score(y_test, y_pred_labels)
 
        if acc > best_acc:
            best_acc = acc
            best_model = model
            best_params = (lr, bs)
 

In [ ]:
y_pred = np.argmax(best_model.predict(X_test), axis=1)
report = classification_report(y_test, y_pred, target_names=class_names)
cm = confusion_matrix(y_test, y_pred)
 
with open(os.path.join(RESULT_DIR, "dnn_performance.txt"), "w") as f:
    f.write("DNN Model Results\n")
    f.write(f"Accuracy: {best_acc}\n")
    f.write(f"Learning rate: {best_params[0]}\n")
    f.write(f"Batch size: {best_params[1]}\n\n")
    f.write(report)
 
best_model.save(os.path.join(RESULT_DIR, "dnn_model.h5"))
 
print("Done. Best accuracy:", best_acc)
print(report)